## sympy triple integration matrices

In [ ]:
from IPython.display import display
import sympy as sp
import jax
import jax.numpy as jnp
import functools
import numpy as np

jax.config.update("jax_enable_x64", True)

In [ ]:
f = sp.Symbol("f", real=True)
s = sp.Symbol("s")

In [ ]:
coeffs_dict = sp.expand((s - f)**3).as_coefficients_dict(s)
coeffs = [coeffs_dict[s**i] for i in range(3)]

In [ ]:
# dt = sp.sympify("0.005")
dt = sp.sympify("1 / 200")
# dt = 0.005

A = sp.Matrix([
    [-coeffs[2], -coeffs[1], -coeffs[0]],
    [1, 0, 0],
    [0, 1, 0],
])
B = sp.Matrix([
    [1],
    [0],
    [0],
])
C = sp.Matrix([[0, 0, (-f)**3]])

Z = sp.ZeroMatrix(*A.shape)
I = sp.Identity(A.shape[0])

bm = sp.Matrix(sp.BlockMatrix([[A, Z], [I, Z]]))
bm_exp = (bm * dt).exp()

y0 = sp.Matrix(sp.BlockMatrix([[I], [Z]]))
E1 = (bm_exp * y0)[A.shape[0]:, :] * B
E1 = sp.simplify(E1)

E0 = sp.simplify((A * dt).exp())

display(E0)
display(E1)

In [ ]:
E0_lam = sp.lambdify([f], E0, modules=["jax"], cse=True, docstring_limit=None)
E1_lam = sp.lambdify([f], E1, modules=["jax"], cse=True, docstring_limit=None)

In [ ]:
print(E0_lam.__doc__)
print()
print(E1_lam.__doc__)

In [ ]:
def fast_trip_E0(f: jax.Array) -> jax.Array:
    x0 = f**2
    x1 = x0 + 80000
    x2 = jnp.exp((1/200)*f)
    x3 = (1/80000)*x2
    x4 = (1/40000)*x2
    x5 = f**3
    x6 = x3*(f + 400)
    return jnp.array([[x3*(800*f + x1), x0*x4*(-f - 600), x5*x6], [x6, x4*(-200*f - x0 + 40000), x3*x5], [x3, x4*(200 - f), x3*(-400*f + x1)]])

def fast_trip_E1(f: jax.Array) -> jax.Array:
    x0 = (1/200)*f
    x1 = jnp.exp(x0)
    x2 = (1/80000)*x1
    return jnp.array([[x2*(f + 400)], [x2], [(f**2*x2 - x0*x1 + x1 - 1)/f**3]])


In [ ]:
def fast_trip_C(f: jax.Array) -> jax.Array:
    return jnp.array([0, 0, (-f)**3])

@jax.jit
def fast_trip_E0_E1_C(f: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
    return fast_trip_E0(f), jnp.ravel(fast_trip_E1(f)), fast_trip_C(f)

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def get_E0_E1_C(den, num, K, nu=0):
    """Integration ZOH scheme."""
    a = jnp.poly(den)
    b = K * jnp.poly(num)
    b = jnp.concatenate([jnp.atleast_1d(b), jnp.zeros(nu)])
    assert a.size - b.size >= 1 + nu
    dt = 1.0 / 200.0

    a_coeffs = a[1:]
    n = a_coeffs.size
    A = jnp.vstack([-a_coeffs, jnp.eye(n - 1, n)])
    B = jnp.zeros(n)
    B = B.at[0].set(1.)
    C = jnp.concatenate([jnp.zeros(n - b.size), b])

    Z = jnp.zeros_like(A)
    I = jnp.eye(*A.shape)  # noqa: E741
    dyn_mat = jnp.block([[A, Z], [I, Z]])
    y0 = jnp.block([[I], [Z]])
    E1 = (jax.scipy.linalg.expm(dyn_mat * dt) @ y0)[A.shape[0] :] @ B
    E0 = jax.scipy.linalg.expm(A * dt)
    return E0, E1, C


@functools.partial(jax.jit, static_argnames=["nu"])
def get_trip_E0_E1_C(root, nu=0):
    den = jnp.ones(3) * root
    num = jnp.array([])
    K = jnp.prod(-den)
    return get_E0_E1_C(den, num, K, nu)

In [ ]:
jnp.max(jnp.concatenate([jnp.ravel(fast_trip_E0_E1_C(-2.0)[idx] - get_trip_E0_E1_C(-2.0)[idx]) for idx in range(3)]))

## observability

In [ ]:
ys = [sp.Symbol(f"y_{i}") for i in range(3)]
us = [sp.Symbol(f"u_{i}") for i in range(2)]

In [ ]:
obs = sp.Matrix(sp.BlockMatrix([[C], [C * E0], [C * E0**2]]))
rhs = sp.Matrix([
    [ys[0]],
    [ys[1] - (C * E1)[0] * us[0]],
    [ys[2] - (C * E0 * E1)[0] * us[0] - (C * E1)[0] * us[1]],
])
obs_op = sp.simplify(obs**-1 * rhs)
obs_op = sp.simplify(E0**2 * obs_op + E0 * E1 * us[0] + E1 * us[1])
obs_op

In [ ]:
obs_lam = sp.lambdify([f, *ys, *us], obs_op, modules=["jax"], cse=True, docstring_limit=None)
print(obs_lam.__doc__)

In [ ]:
@jax.jit
def fast_obs_x0(f, y_0, y_1, y_2, u_0, u_1):
    x0 = f**3
    x1 = x0**(-1.0)
    x2 = f*y_2
    x3 = f**2
    x4 = jnp.exp((1/200)*f)
    x5 = jnp.exp((1/100)*f)
    x6 = x5*y_0
    x7 = 160000*x4
    x8 = f*u_0
    x9 = x3*x4
    return jnp.ravel(jnp.array([[(1/400)*x1*(80000*f*u_0*x5 - f*u_1*x7 + 240000*f*u_1 + 320000*f*x4*y_1 - 80000*f*x6 - u_0*x0*x4 - 16000000*u_0*x4 + 16000000*u_0*x5 - 600*u_0*x9 + u_1*x0*x4 + 600*u_1*x3*x4 + 400*u_1*x3 - 16000000*u_1*x4 + 16000000*u_1 - 240000*x2 - 400*x3*y_2 + 32000000*x4*y_1 - 16000000*x6 - x7*x8 - 16000000*y_2)], [x1*((1/2)*f*u_1*x4 + f*u_1 - 100*u_0*x4 + 100*u_0*x5 - 1/800*u_0*x9 + (1/800)*u_1*x3*x4 - 300*u_1*x4 + 300*u_1 - x2 - 1/2*x4*x8 + 400*x4*y_1 - 100*x6 - 300*y_2)], [-x1*y_2]]))


In [ ]:
f_eval = -5.0
C_np = np.ravel(np.array(C.subs(f, f_eval).evalf(), dtype=np.float64))
E0_np = np.array(E0.subs(f, f_eval).evalf(), dtype=np.float64)
E1_np = np.ravel(np.array(E1.subs(f, f_eval).evalf(), dtype=np.float64))

np.random.seed(0)
us_np = np.random.uniform(0.0, 1.0, size=10)
x0 = np.random.uniform(-1.0, 1.0, size=3)
ys_np = np.empty(11)
ys_np[0] = C_np @ x0
for i in range(us_np.size):
    x0 = E0_np @ x0 + E1_np * us_np[i]
    ys_np[i + 1] = C_np @ x0

In [ ]:
fast_obs_x0(f_eval, ys_np[-3], ys_np[-2], ys_np[-1], us_np[-2], us_np[-1]) - x0